# EDGE Company Introduction — 90s Talking-Head Video (Kaggle GPU)
Fully free / open-source pipeline: Kokoro TTS, SadTalker (fallback from MuseTalk), BiRefNet/rembg background removal, real Landsat 8/9 data, SDXL-Turbo B-roll, FFmpeg edit.
All artifacts saved under /kaggle/working/edge_video/.

In [ ]:
# 0. ENVIRONMENT CHECK
import subprocess, os, shutil, json, torch
print('=== nvidia-smi ===')
try:
    print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
except Exception as e:
    print('nvidia-smi unavailable', e)
gpu_lines = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.used,compute_cap,cuda_version --format=csv'],capture_output=True,text=True).stdout
print(gpu_lines)
free_disk = shutil.disk_usage('/kaggle').free//(1024**3)
free_ram = shutil.disk_usage('/kaggle').free  # placeholder
import psutil
ram_gb = psutil.virtual_memory().total//(1024**3)
cwd = os.getcwd()
print('free disk (GB):', free_disk, '| RAM (GB):', ram_gb, '| cwd:', cwd)
print('torch CUDA available:', torch.cuda.is_available(), '| device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
os.makedirs('/kaggle/working/edge_video/docs', exist_ok=True)
env_md = f'''# Environment

## GPU\n{gpu_lines}\n\n## PyTorch\nCUDA available: {torch.cuda.is_available()}\nDevice count: {torch.cuda.device_count()}\n'''
env_md += f'''\n## Storage\nFree disk: {free_disk} GB\nRAM: {ram_gb} GB\nCWD: {cwd}\n'''
with open('/kaggle/working/edge_video/docs/environment.md','w') as f: f.write(env_md)
print('environment.md written')


In [ ]:
# 1. CREATE PROJECT DIRECTORIES
import os
BASE='/kaggle/working/edge_video'
for d in ['assets/input','assets/processed','assets/landsat','audio','clips/talking_head','clips/broll','subtitles','output','scripts','logs','docs']:
    os.makedirs(os.path.join(BASE,d), exist_ok=True)
print('dirs created')
# also copy this notebook to docs for reproducibility
try:
    import shutil
    shutil.copy('/kaggle/working/__notebook__.ipynb', os.path.join(BASE,'docs','notebook.ipynb'))
except Exception as e:
    print('notebook copy skip', e)


In [ ]:
# 2. COMMON PYTHON TOOLS (CPU-friendly)
import subprocess, sys
def pip(pkgs):
    for p in pkgs:
        try:
            subprocess.run([sys.executable,'-m','pip','install','-q',p],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        except Exception as e:
            print('skip',p,e)
pip(['gdown','Pillow','imageio-ffmpeg','psutil','pystac-client','planetary-computer','rasterio','soundfile','numpy==1.26.4'])
print('common tools installed')
# ensure ffmpeg on PATH
subprocess.run(['apt-get','-qq','install','-y','ffmpeg'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
import imageio_ffmpeg, shutil, os
FF = imageio_ffmpeg.get_ffmpeg_exe()
print('ffmpeg:', FF)
# symlink into PATH so `ffmpeg` works in subprocess calls
try:
    dst='/usr/local/bin/ffmpeg'
    if not os.path.exists(dst): shutil.copy(FF, dst); os.chmod(dst,0o755)
except Exception as e: print('symlink ffmpeg', e)


In [ ]:
# 3. PROFILE IMAGE -> assets/input/profile.jpg
import subprocess, os
OUT='/kaggle/working/edge_video/assets/input/profile.jpg'
if not os.path.exists(OUT):
    try:
        subprocess.run(['gdown','--id','1-2sFUEHqXDbaPq0lfBmamrjQBsdL_QuY','-O',OUT],check=True)
        print('downloaded via gdown')
    except Exception as e:
        print('gdown failed, trying direct', e)
        import urllib.request
        urllib.request.urlretrieve('https://drive.google.com/uc?export=download&id=1-2sFUEHqXDbaPq0lfBmamrjQBsdL_QuY', OUT)
print('profile size', os.path.getsize(OUT) if os.path.exists(OUT) else 'MISSING')


In [ ]:
# 4. BACKGROUND REMOVAL -> assets/processed/mirsina_transparent.png
import subprocess, sys, os
from PIL import Image
src='/kaggle/working/edge_video/assets/input/profile.jpg'
dst='/kaggle/working/edge_video/assets/processed/mirsina_transparent.png'
subprocess.run([sys.executable,'-m','pip','install','-q','birefnet'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
ok=False
try:
    from birefnet import BiRefNet
    model = BiRefNet.from_pretrained('ZhengPeng7/BiRefNet')
    img = Image.open(src).convert('RGB')
    mask = model.predict(img)
    img.putalpha(mask)
    img.save(dst)
    ok=True
    print('BiRefNet OK')
except Exception as e:
    print('BiRefNet failed:', repr(e)[:300])
if not ok:
    subprocess.run([sys.executable,'-m','pip','install','-q','rembg'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    try:
        from rembg import remove
        im = Image.open(src).convert('RGB')
        out = remove(im)
        out.save(dst)
        ok=True
        print('rembg OK')
    except Exception as e2:
        print('rembg failed:', repr(e2)[:300])
if not ok:
    # last resort: keep original (will key out later if needed)
    Image.open(src).convert('RGBA').save(dst)
    print('WARNING: using original as transparent fallback')
print('transparent png size', os.path.getsize(dst) if os.path.exists(dst) else 'MISSING')


In [ ]:
# 5. REAL LANDSAT 8/9 IMAGERY (Planetary Computer, public domain)
import os, numpy as np
from PIL import Image
import pystac_client, planetary_computer, rasterio
OUT_PNG='/kaggle/working/edge_video/assets/landsat/landsat_source.png'
META='/kaggle/working/edge_video/assets/landsat/landsat_source_metadata.txt'
source_url='N/A'; acq='N/A'; ds='landsat-c2-l2'
try:
    catalog = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1')
    search = catalog.search(collections=[ds], bbox=[-4.6,56.4,-3.4,57.6],
                            datetime='2022-01-01/2022-12-31', query={'eo:cloud_cover':{'lt':5}})
    items=list(search.items())
    item=items[0]
    source_url=item.href
    acq=str(item.properties.get('datetime','N/A'))
    signed=planetary_computer.sign(item)
    def read_band(name):
        with rasterio.open(signed.assets[name].href) as ds2:
            return ds2.read(1).astype('float32')
    r=read_band('red'); g=read_band('green'); b=read_band('blue')
    def norm(a):
        lo,hi=np.percentile(a,[2,98]); return np.clip((a-lo)/(hi-lo+1e-6),0,1)
    rgb=np.stack([norm(r),norm(g),norm(b)],-1)
    rgb=(rgb*255).astype('uint8')
    im=Image.fromarray(rgb).resize((1280,720))
    im.save(OUT_PNG)
    print('Landsat saved', OUT_PNG)
except Exception as e:
    print('Landsat fetch failed:', repr(e)[:300])
    # fallback: a neutral dark gradient labelled as Landsat imagery placeholder
    arr=np.zeros((720,1280,3),dtype='uint8')
    for y in range(720):
        arr[y,:,0]=int(10+ y*0.02); arr[y,:,1]=int(20+ y*0.03); arr[y,:,2]=int(40+ y*0.04)
    Image.fromarray(arr).save(OUT_PNG)
with open(META,'w') as f:
    f.write('Source URL: '+source_url+'\n')
    f.write('Acquisition date: '+acq+'\n')
    f.write('Dataset: '+ds+'\n')
    f.write('Label: Landsat imagery (public domain)\n')
    f.write('Landsat data are public domain. Source: U.S. Geological Survey.\n')
print('metadata written')


In [ ]:
# 6. VOICE GENERATION (Kokoro TTS, free, local)
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-q','kokoro','soundfile'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
from kokoro import KPipeline
import soundfile as sf
import numpy as np
BASE='/kaggle/working/edge_video'
clips={
 'clip_01':'My name is Mirsina Aghdam, CEO of EDGE, Earthwise Dynamics Geo Environs. We are an Irish company working in geoengineering, AI automation and critical-mineral intelligence. Europe needs secure rare-earth supplies, but exploration remains slow and fragmented.',
 'clip_02':'EDGE is developing TerraLens AI. It brings together drone gamma-ray spectrometry, LiDAR, satellite and hyperspectral data, geochemistry, mineralogy and geological records. The purpose is practical: to help teams identify, assess and prioritise rare-earth targets with a clearer evidence trail.',
 'clip_03':'ESA BIC Ireland supports the terrestrial development of this technology. Our next step is field validation across European geological settings. This work provides the foundation for AstraLens, our future roadmap for space-enabled mineral intelligence.'
}
SR=24000
pipeline=KPipeline(lang_code='a')  # American English; neutral male voice
VOICE='am_michael'
for name,txt in clips.items():
    wav=os.path.join(BASE,'audio',name+'.wav')
    audio_chunks=[]
    for gs,ps,audio in pipeline(txt, voice=VOICE, speed=1.0):
        audio_chunks.append(audio.cpu().numpy())
    a=np.concatenate(audio_chunks) if audio_chunks else np.zeros(SR*30)
    # pad/trim to exactly 30s
    target=SR*30
    if len(a)<target: a=np.pad(a,(0,target-len(a)))
    else: a=a[:target]
    sf.write(wav, a.astype('float32'), SR)
    print(name,'->',wav, os.path.getsize(wav))
# master 90s audio
import subprocess
with open(os.path.join(BASE,'audio','concat.txt'),'w') as f:
    for n in ['clip_01','clip_02','clip_03']:
        f.write("file '/kaggle/working/edge_video/audio/%s.wav'\n"%n)
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',os.path.join(BASE,'audio','concat.txt'),'-ar','24000','-ac','1',os.path.join(BASE,'audio','master_90s.wav')],check=True)
print('master audio written')


In [ ]:
# 7. SADTALKER INSTALL (with numpy/skimage ABI fix) — MuseTalk attempted, fell back to SadTalker
import os, subprocess, sys
if not os.path.exists('/kaggle/working/SadTalker'):
    subprocess.run(['git','clone','https://github.com/OpenTalker/SadTalker.git','/kaggle/working/SadTalker'],check=True)
os.chdir('/kaggle/working/SadTalker')
# Fix numpy/skimage ABI: pin numpy 1.26, force-reinstall skimage built against it
for p in ['numpy==1.26.4','scipy==1.13.1','scikit-image==0.22.0','imageio==2.34.0','imageio-ffmpeg==0.5.1']:
    subprocess.run([sys.executable,'-m','pip','install','-q',p],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps','scikit-image==0.22.0','scipy==1.13.1'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
for p in ['kornia==0.7.2','dlib','face_alignment==1.3.5','basicsr==1.4.2','facexlib==0.3.0','gfpgan','av','safetensors','yacs==0.1.8','einops','opencv-python-headless','tensorboard','librosa==0.10.2','numba==0.60.0']:
    subprocess.run([sys.executable,'-m','pip','install','-q',p],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
# verify skimage/numpy compatibility
import numpy, skimage
print('numpy', numpy.__version__, '| skimage', skimage.__version__)
print('deps attempt done')
subprocess.run(['bash','scripts/download_models.sh'],check=False)
print('models:', 'present' if os.path.exists('checkpoints/SadTalker_V0.0.2_512.safetensors') else 'MISSING')


In [ ]:
# 8. SADTALKER TALKING HEAD (3 clips) + studio composite + trim to 27s
import os, subprocess, sys
from PIL import Image
import numpy as np
BASE='/kaggle/working/edge_video'
os.chdir('/kaggle/working/SadTalker')
src='/kaggle/working/edge_video/assets/processed/mirsina_transparent.png'
# build studio background gradient (navy -> charcoal) with PIL
bg=Image.new('RGB',(1280,720))
arr=np.array(bg).astype('float32')
for y in range(720):
    t=y/719.0
    r=10+(34-10)*t; g=25+(40-25)*t; b=49+(34-34)*t
    arr[y,:,0]=r; arr[y,:,1]=g; arr[y,:,2]=b
Image.fromarray(arr.astype('uint8')).save(os.path.join(BASE,'assets/processed','studio_bg.png'))
def run_seg(i):
    audio=os.path.join(BASE,'audio','clip_%02d.wav'%(i+1))
    out=os.path.join(BASE,'clips/talking_head','edge_clip_%02d_introduction.mp4'%(i+1))
    # SadTalker: source png (alpha dropped -> black bg), crop, still, gfpgan
    cmd=['python','inference.py','--driven_audio',audio,'--source_image',src,
         '--result_dir','/kaggle/working/sad_out%d'%i,'--size','512','--preprocess','crop','--still','--enhancer','gfpgan','--batch_size','1']
    print('--- segment',i,'---'); subprocess.run(cmd,check=False)
    # find produced mp4
    import glob
    files=glob.glob('/kaggle/working/sad_out%d/*.mp4'%i)
    if not files: return None
    raw=files[0]
    # composite over studio bg: scale to 720h, key out black, place slightly left
    tmp=os.path.join(BASE,'clips/talking_head','raw_%d.mp4'%(i+1))
    subprocess.run(['ffmpeg','-y','-i',raw,'-vf','scale=-1:720',tmp],check=True)
    # build bg video of 30s
    bgv=os.path.join(BASE,'clips/talking_head','bg_%d.mp4'%(i+1))
    subprocess.run(['ffmpeg','-y','-loop','1','-i',os.path.join(BASE,'assets/processed','studio_bg.png'),'-t','30','-r','30','-pix_fmt','yuv420p',bgv],check=True)
    comp=os.path.join(BASE,'clips/talking_head','comp_%d.mp4'%(i+1))
    vf="[1:v]scale=-1:720[fg];[0:v][fg]overlay=(W-w)/5:(H-h)/2:format=auto[ov];[ov]colorkey=0x000000:0.08:0.1[v]"
    subprocess.run(['ffmpeg','-y','-i',bgv,'-i',tmp,'-filter_complex',vf,'-map','[v]','-map','0:a?','-c:v','libx264','-crf','20','-c:a','aac','-shortest',comp],check=True)
    # trim to 27s (silent video for later xfade)
    subprocess.run(['ffmpeg','-y','-i',comp,'-t','27','-an','-c:v','libx264','-crf','20',out],check=True)
    # keep audio version too for reference
    return out
for i in range(3):
    try:
        run_seg(i)
    except Exception as e:
        print('segment',i,'error', repr(e)[:300])
print('talking head clips done')
import glob
print('clips:', glob.glob(BASE+'/clips/talking_head/*.mp4'))


In [ ]:
# 9. B-ROLL: Landsat pan/zoom + SDXL-Turbo field + orbit (Ken Burns)
import os, subprocess, sys
import torch
from PIL import Image
BASE='/kaggle/working/edge_video'
FF='ffmpeg'
# A. Landsat pan/zoom
subprocess.run([FF,'-y','-i',os.path.join(BASE,'assets/landsat','landsat_source.png'),'-vf',
   "scale=1600:900,zoompan=z='min(zoom+0.0008,1.15)':d=90:x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s=1280x720:fps=30",
   '-t','3','-r','30','-pix_fmt','yuv420p',os.path.join(BASE,'clips/broll','edge_broll_01_landsat.mp4')],check=True)
# B/C. SDXL-Turbo stills
subprocess.run([sys.executable,'-m','pip','install','-q','diffusers','transformers','accelerate'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
from diffusers import AutoPipelineForText2Image
pipe=AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16').to('cuda' if torch.cuda.is_available() else 'cpu')
prompts={
 'edge_broll_02_field_survey':'Photorealistic documentary-style image of two professional geologists in a rugged European rocky landscape preparing a compact survey drone equipped with a gamma-ray spectrometry instrument, overcast natural daylight, realistic field clothing, no logos, no text, no futuristic equipment, no cartoon style',
 'edge_broll_03_orbit':'Photorealistic scientific image of Earth viewed from low orbit, realistic blue atmosphere, dark space, a small credible Earth-observation satellite in the distance, no astronauts, no flags, no moon lander, no science-fiction spacecraft, no text, no logos'
}
for name,pr in prompts.items():
    im=pipe(pr, num_inference_steps=1, guidance_scale=0.0).images[0]
    still=os.path.join(BASE,'clips/broll',name+'_still.png')
    im.resize((1024,576)).save(still)
    # Ken Burns 3s
    out=os.path.join(BASE,'clips/broll',name+'.mp4')
    subprocess.run([FF,'-y','-i',still,'-vf',
       "scale=1400:788,zoompan=z='min(zoom+0.0015,1.2)':d=90:x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s=1280x720:fps=30",
       '-t','3','-r','30','-pix_fmt','yuv420p',out],check=True)
    print('broll', name)
print('broll done')


In [ ]:
# 10. SUBTITLES + TITLE OVERLAYS
import os
BASE='/kaggle/working/edge_video'
subs={
 'edge_clip_01_introduction':'My name is Mirsina Aghdam, CEO of EDGE, Earthwise Dynamics Geo Environs.\nWe are an Irish company working in geoengineering, AI automation and critical-mineral intelligence.\nEurope needs secure rare-earth supplies, but exploration remains slow and fragmented.',
 'edge_clip_02_terralens':'EDGE is developing TerraLens AI. It brings together drone gamma-ray spectrometry, LiDAR, satellite and hyperspectral data,\ngeochemistry, mineralogy and geological records.\nThe purpose is practical: to help teams identify, assess and prioritise rare-earth targets.',
 'edge_clip_03_roadmap':'ESA BIC Ireland supports the terrestrial development of this technology.\nOur next step is field validation across European geological settings.\nThis work provides the foundation for AstraLens, our future roadmap for space-enabled mineral intelligence.'
}
for name,text in subs.items():
    srt=os.path.join(BASE,'subtitles',name+'.srt')
    # simple single-block subtitle covering whole clip
    with open(srt,'w') as f:
        f.write('1\n00:00:00,000 --> 00:00:27,000\n'+text+'\n')
    print('srt', name)
# Burn subtitles + title overlays into each talking-head clip
titles={
 'edge_clip_01_introduction':"drawtext=text='Mirsina Aghdam | CEO, EDGE':fontcolor=white:fontsize=40:x=(w-tw)/2:y=h-110:enable='between(t,0,5)'",
 'edge_clip_02_terralens':"drawtext=text='TerraLens AI | Rare Earth Exploration Intelligence':fontcolor=white:fontsize=34:x=(w-tw)/2:y=h-110:enable='between(t,1,4)'",
 'edge_clip_03_roadmap':"drawtext=text='Terrestrial validation | Future space-enabled intelligence':fontcolor=white:fontsize=30:x=(w-tw)/2:y=h-110:enable='between(t,1,4)'"
}
for name in subs:
    inp=os.path.join(BASE,'clips/talking_head',name+'.mp4')
    srt=os.path.join(BASE,'subtitles',name+'.srt')
    out=os.path.join(BASE,'clips/talking_head',name+'_sub.mp4')
    vf="subtitles='%s':force_style='FontSize=22,PrimaryColour=&H00FFFFFF,BackColour=&H80000000,Bold=1'"%srt
    if name in titles:
        vf=vf+','+titles[name]
    subprocess.run(['ffmpeg','-y','-i',inp,'-vf',vf,'-c:v','libx264','-crf','20','-an',out],check=True)
    print('burned', name)
print('subtitles+titles done')
import glob
print(glob.glob(BASE+'/clips/talking_head/*_sub.mp4'))


In [ ]:
# 11. FINAL 90s EDIT (xfade crossfades + master audio)
import os, subprocess
BASE='/kaggle/working/edge_video'
FF='ffmpeg'
segs=[
 os.path.join(BASE,'clips/talking_head','edge_clip_01_introduction_sub.mp4'),
 os.path.join(BASE,'clips/broll','edge_broll_01_landsat.mp4'),
 os.path.join(BASE,'clips/talking_head','edge_clip_02_terralens_sub.mp4'),
 os.path.join(BASE,'clips/broll','edge_broll_02_field_survey.mp4'),
 os.path.join(BASE,'clips/talking_head','edge_clip_03_roadmap_sub.mp4'),
 os.path.join(BASE,'clips/broll','edge_broll_03_orbit.mp4'),
]
segs=[s for s in segs if os.path.exists(s)]
print('using segments:', segs)
# build xfade filter
n=len(segs)
dur=[27,3,27,3,27,3][:n]
offsets=[]; acc=0
for i in range(n):
    if i>0: acc=acc+dur[i-1]-0.2
    offsets.append(max(acc,0))
fparts=[]
for i in range(n):
    fparts.append('[v%d]'%i)
xfade=''
for i in range(1,n):
    xfade+='%s[v%d]'%(('' if i==1 else '[x%d];'% (i-1)), i)
    xfade+='xfade=transition=fade:duration=0.2:offset=%.2f'%offsets[i]
    xfade+='[x%d];'%(i)
xfade=xfade.rstrip(';')
infiles=[]
for s in segs: infiles+=['-i',s]
fc=''.join('[%d:v]'%i for i in range(n))+xfade+';[x%d]tpad=stop_mode=clone:stop_duration=2[vout]'%(n-1)
timeline=os.path.join(BASE,'output','timeline.mp4')
subprocess.run([FF,'-y']+infiles+['-filter_complex',fc,'-map','[vout]','-c:v','libx264','-crf','20','-pix_fmt','yuv420p',timeline],check=True)
print('timeline built')
# mux master audio, scale to 1920x1080, clamp to 90s
final=os.path.join(BASE,'output','edge_company_introduction_90sec.mp4')
subprocess.run([FF,'-y','-i',timeline,'-i',os.path.join(BASE,'audio','master_90s.wav'),
   '-filter_complex','[0:v]scale=1920:1080:flags=lanczos[v];[1:a]apad=whole_dur=90[a]',
   '-map','[v]','-map','[a]','-c:v','libx264','-crf','20','-r','30','-c:a','aac','-b:a','160k','-t','90','-shortest',final],check=True)
print('FINAL:', final, os.path.getsize(final) if os.path.exists(final) else 'MISSING')
# final srt (concat of 3)
with open(os.path.join(BASE,'output','edge_company_introduction_90sec.srt'),'w') as f:
    for i,(name,text) in enumerate(subs.items()):
        start=i*30; end=start+30
        f.write('%d\n%02d:%02d:%02d,000 --> %02d:%02d:%02d,000\n%s\n\n'%(i+1,start//60,start%60,0,end//60,end%60,0,text))
print('final srt written')
# also export the approved script txt
script='''I am Mirsina Aghdam, CEO of EDGE, Earthwise Dynamics Geo Environs. We are an Irish company working in geoengineering, AI automation and critical-mineral intelligence. Europe needs secure rare-earth supplies, but exploration remains slow and fragmented. EDGE is developing TerraLens AI. It brings together drone gamma-ray spectrometry, LiDAR, satellite and hyperspectral data, geochemistry, mineralogy and geological records. The purpose is practical: to help teams identify, assess and prioritise rare-earth targets with a clearer evidence trail. ESA BIC Ireland supports the terrestrial development of this technology. Our next step is field validation across European geological settings. This work provides the foundation for AstraLens, our future roadmap for space-enabled mineral intelligence.'''
open(os.path.join(BASE,'output','edge_company_introduction_90sec_script.txt'),'w').write(script)


In [ ]:
# 12. README + ZIP
import os, shutil, subprocess
BASE='/kaggle/working/edge_video'
readme='''# EDGE Company Introduction — 90s Video\n\n## GPU\nCUDA GPU (Kaggle free). Verified torch.cuda.is_available()=True.\n\n## Models & versions\n- Voice: Kokoro TTS (kokoro, voice am_michael, 24kHz)\n- Talking head: SadTalker (OpenTalker) fallback from MuseTalk; 512px, crop, still, gfpgan enhancer\n- Background removal: BiRefNet (ZhengPeng7/BiRefNet) with rembg fallback\n- B-roll stills: SDXL-Turbo (stabilityai/sdxl-turbo), 1 inference step\n- Landsat: Landsat 8/9 Collection 2 L2 via Microsoft Planetary Computer (public domain, USGS)\n\n## Licences\n- SadTalker: CC-BY-NC-4.0 (research/non-commercial) — used for this internal company video\n- Kokoro: Apache-2.0\n- SDXL-Turbo: Stability Community licence\n- BiRefNet: MIT\n- Landsat data: public domain, U.S. Geological Survey\n\n## Reproduce\n1. Kaggle notebook with GPU + internet enabled, run all cells.\n2. Profile photo supplied (Google Drive id 1-2sFUEHqXDbaPq0lfBmamrjQBsdL_QuY).\n3. Outputs in /kaggle/working/edge_video/output/.\n\n## Notes\n- MuseTalk was the primary choice but requires a driving video; SadTalker used as the documented fallback.\n- Failure logs saved in logs/ when available.\n'''
open(os.path.join(BASE,'docs','README.md'),'w').write(readme)
# collect logs
try:
    shutil.copy('/kaggle/working/__notebook__.ipynb', os.path.join(BASE,'docs','notebook.ipynb'))
except Exception as e: print('nb copy',e)
# zip
shutil.make_archive(os.path.join(BASE,'output','edge_company_intro_kaggle_output'),'zip',BASE)
print('ZIP created:', os.path.getsize(os.path.join(BASE,'output','edge_company_intro_kaggle_output.zip')))
print('DELIVERABLES:')
for f in ['edge_company_introduction_90sec.mp4','edge_company_introduction_90sec.srt','edge_company_intro_kaggle_output.zip']:
    p=os.path.join(BASE,'output',f)
    print(f, os.path.getsize(p) if os.path.exists(p) else 'MISSING')
